# Member Attrition Model Development

This historical portfolio notebook engineers monthly behavioral features and compares several classification approaches for predicting member attrition, including logistic regression, random forest, and XGBoost.

Live database access, organization-specific infrastructure, and original outputs have been removed. The notebook expects anonymized local extracts and is preserved primarily to demonstrate the modeling workflow and iterative feature-development process.


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt  
%matplotlib inline 
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', lambda x: '%.2f' % x)

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 115)

# Transactions Data

In [ ]:
from pathlib import Path

TRANSACTION_FILE = Path('member_transactions.csv')
if not TRANSACTION_FILE.exists():
    raise FileNotFoundError('Place an anonymized member_transactions.csv file in this directory.')
sq_df = pd.read_csv(TRANSACTION_FILE, parse_dates=['PostDate'])


### Member Data - Used to create churn variable

In [ ]:
from pathlib import Path

MEMBER_FILE = Path('member_status_history.csv')
if not MEMBER_FILE.exists():
    raise FileNotFoundError('Place an anonymized member_status_history.csv file in this directory.')
closed_sq_df = pd.read_csv(MEMBER_FILE, parse_dates=['OpenDate', 'CloseDate'])


In [ ]:
sq_df.head()

In [ ]:
sq_df.PostDate.min()

In [ ]:
sq_df.PostDate.max()

In [ ]:
#Subscription Deposits flag direct deposit changes
#infomration provided by the business team for these business rules
dfsubd1 = sq_df.loc[(sq_df.ActionCodeDescription == 'Deposit')&(sq_df.TranSourceCd.isin(['B','E','G','H','O','T']))]
 #dfsubd1 = dfsubd1[dfsubd1.duplicated(subset=['member_id','Description','PrimaryDescription','TranAmount'], keep=False)]
dfsubd1 = dfsubd1.sort_values(['member_id','Description','PrimaryDescription','TranAmount'])

#added features
cols = [dfsubd1['member_id'], dfsubd1['Description'], dfsubd1['PrimaryDescription'], dfsubd1['TranAmount']]
#deposits - this looks at weekly/bi-weekly/monthly cadence of deposit subscriptions
dfsubd1 = dfsubd1[dfsubd1['PostDate'].groupby(cols).diff().dt.days.fillna(1).isin([5,6,7,8,13,14,15,16,28,29,30,31])]
dfsubd1['DirectDeposit'] = 1 
dfsubd1 = dfsubd1.drop_duplicates(['member_id','TranAmount','ActionCodeDescription','Description','DirectDeposit','ProductCategory'])
#deposits $500+
dfsubd500 = dfsubd1.loc[(dfsubd1.TranAmount >= 500)]
dfsubd500['DirectDeposit500'] = 1 
dfsubd500 = dfsubd500.drop_duplicates(['member_id','TranAmount','ActionCodeDescription','Description','DirectDeposit','ProductCategory'])

sq_df1 = pd.merge(sq_df,dfsubd1[['member_id','TranAmount','ActionCodeDescription','Description','DirectDeposit','ProductCategory']],
                  how='left',on=['member_id','TranAmount','ActionCodeDescription','Description','ProductCategory'])

sq_df2 = pd.merge(sq_df1,dfsubd500[['member_id','TranAmount','ActionCodeDescription','Description','DirectDeposit500','ProductCategory']],
                  how='left',on=['member_id','TranAmount','ActionCodeDescription','Description','ProductCategory'])

sq_df3 = sq_df2.fillna(0)

#subscription accounts for FAANG type
sq_df3.loc[sq_df3['PrimaryDescription'].str.contains('Apple|Amazon|Netflix|Hulu|HBO|Spotify'
                                                     ,case=False, na=False),'CardSubFAANG'] = 1

#credit limit usage percent
sq_df3.loc[sq_df3['LoanTypeCategory'].str.contains('Credit Card',case=False, na=False)
           ,'CCLimitUsage'] = sq_df3['NewBalance']/sq_df3['LoanCreditLimit']

#branch visits
sq_df3.loc[sq_df3['Channel'].str.contains('Branch',case=False, na=False)
           ,'BranchVisit'] = 1

#Bill Pay
sq_df3.loc[(sq_df3['Channel'].str.contains('Bill Pay|BillPay')), 'BillPay'] = 1

sq_df3 = sq_df3.fillna(0)


In [ ]:
sq_df3.head()

In [ ]:
#churn variables
closed_sq_df['ClosedMonth'] = closed_sq_df['CloseDate'].apply(lambda x: x.strftime('%Y-%m-%d')).str[:7]
closed_sq_df['ClosedMonthSid'] = closed_sq_df.ClosedMonth.str.replace('-', '').str[0:6]
closed_sq_df['ClosedMonth'] = pd.to_datetime(closed_sq_df['ClosedMonth']).dt.to_period('M')
closed_sq_df['Churn'] = 1 
closed_sq_df.head()

In [ ]:
#create variables for prior month to churn ... i.e. churn in 1 month, used for prediction of churn
closed_sq_df['PriorClosedMonth'] = closed_sq_df.ClosedMonth - 1
closed_sq_df['PriorClosedMonthSid'] = [''.join(x.split('-')[0:2]) for x in closed_sq_df.PriorClosedMonth.astype(str)]
closed_sq_df['PriorClosedMonthSid'] = closed_sq_df['PriorClosedMonthSid'].astype(int)
closed_sq_df['ChurnIn1Mo'] = closed_sq_df['Churn']
closed_sq_df.head()

In [ ]:
sq_df3['PostDateSid'] = sq_df3['PostDateSid'].apply(str)
sq_df3['yearmonthsid'] = sq_df3['PostDateSid'].str[:6]

In [ ]:
df = sq_df3.copy()
df = df.sort_values(by=['member_id','PostDate'])

In [ ]:
df.shape

In [ ]:
#calculate time difference between events ... i.e. recency
df['Time_diff_Product'] = df.groupby(['member_id','ShareTypeCategory'])['PostDate'].diff() / np.timedelta64(1, 'D')
df['Time_diff'] = df.groupby(['member_id'])['PostDate'].diff() / np.timedelta64(1, 'D')
df['Time_diff_Product'] = df['Time_diff'].fillna(0)
df['Time_diff'] = df['Time_diff'].fillna(0)


In [ ]:
#create aggregation tables of transaction data at a monthly level
df['PostDateMonth'] = pd.to_datetime(df['PostDate']).dt.to_period('M')

#share aggregation table
cols = ['member_id','PostDateMonth','ProductCategory','ShareTypeName','ShareTypeCategory',\
        'TranActionCd','SourceCodeDescription']
dfgs = df.groupby(cols).agg({'member_id':'count',
                             'BalanceChange':'sum',
                             'PrevAvailBalance':'last',
                             'FeeAmount':'sum',
                             'Interest':'sum',
                             'DirectDeposit':'sum',
                             'DirectDeposit500':'sum',
                             'BillPay':'sum',
                             'CardSubFAANG':'sum',
                             'BranchVisit':'sum',
                             'Time_diff_Product':'max',
                             'Time_diff':'max',
                             'Churn':'sum'})

#loan aggregation table
coll = ['member_id','PostDateMonth','ProductCategory',\
       'LoanTypeName','LoanTypeCategory','TranActionCd','SourceCodeDescription']
dfgl = df.groupby(coll).agg({'member_id':'count',
                             'BalanceChange':'sum',
                             'PrevAvailBalance':'last',
                             'FeeAmount':'sum',
                             'Interest':'sum',
                             'BillPay':'sum',
                             'CCLimitUsage':'max',
                             'CardSubFAANG':'sum',
                             'BranchVisit':'sum',
                             'Time_diff_Product':'max',
                             'Time_diff':'max',
                             'Churn':'sum'})

In [ ]:
dfgl.head()

In [ ]:
dfgs1 = dfgs.rename(columns = {'member_id':'ActivityCount'}).reset_index()


In [ ]:
dfgs1.head()

In [ ]:
dfgs1.shape

In [ ]:
#separate out churn dataframe
cols1 = ['member_id','PostDateMonth']
memdfchurn = df.groupby(cols1).agg({'Churn':'sum'})
memdfchurn = memdfchurn.reset_index()
memdfchurn.loc[memdfchurn.Churn > 1, 'Churn'] = 1


#unstack multi-layer member_id
dfgs1 = dfgs.rename(columns = {'member_id':'ActivityCount'}).reset_index()
dfgs1 = dfgs1.loc[dfgs1['ShareTypeCategory'] != 0]

#create share type catagory-action column
dfgs1['ShareTypeCatAction'] = dfgs1[['ShareTypeCategory','TranActionCd']].agg('-'.join,axis=1)

#looking for "top of wallet" debit card per Darren's definition
dfgs1.loc[((dfgs1.SourceCodeDescription.str.contains('Debit')) & (dfgs1.ShareTypeCategory.str.contains('Checking'))
           & (dfgs1.ActivityCount > 9)),'TopDebit'] = 1

#creating new features for direct deposit and usage over time
dfgs1['DirectDepositLastMo']=dfgs1.groupby(['member_id','ShareTypeCategory','TranActionCd','SourceCodeDescription']).apply(lambda x :x.DirectDeposit.shift()* (x.PostDateMonth.dt.year*12+x.PostDateMonth.dt.month).diff().eq(1)).replace(0,np.nan).values
dfgs1['DirectDeposit500LastMo']=dfgs1.groupby(['member_id','ShareTypeCategory','TranActionCd','SourceCodeDescription']).apply(lambda x :x.DirectDeposit500.shift()* (x.PostDateMonth.dt.year*12+x.PostDateMonth.dt.month).diff().eq(1)).replace(0,np.nan).values
dfgs1['BillPayLastMo']=dfgs1.groupby(['member_id','ShareTypeCategory','TranActionCd','SourceCodeDescription']).apply(lambda x :x.BillPay.shift()* (x.PostDateMonth.dt.year*12+x.PostDateMonth.dt.month).diff().eq(1)).replace(0,np.nan).values
dfgs1.loc[((dfgs1.DirectDeposit==0)&(dfgs1.DirectDepositLastMo==1)),'DirectDepositStop'] = 1

newcols = ['member_id', 'PostDateMonth','ShareTypeCatAction', 'BalanceChange', 'PrevAvailBalance',
          'FeeAmount', 'Interest','TopDebit','DirectDeposit','DirectDeposit500','DirectDepositLastMo',
           'DirectDeposit500LastMo','BillPayLastMo','DirectDepositStop','Time_diff_Product','Time_diff','Churn']

#unstack and aggregate by member_id, PostDateMonth, ShareTypeCategory, TranActionCd
memdfs = dfgs1[newcols]
memdfs = memdfs.pivot_table(index=['member_id','PostDateMonth'],
                            columns = ['ShareTypeCatAction'],
                            values=['BalanceChange','PrevAvailBalance','FeeAmount','Interest','DirectDeposit',
                                    'DirectDeposit500','DirectDepositLastMo', 'DirectDeposit500LastMo','BillPayLastMo',
                                    'DirectDepositStop','Time_diff_Product','Time_diff'],
                            aggfunc=np.sum) 
    
#reset index and rename columns
memdfs = memdfs.reset_index()
memdfs.columns = ['_'.join(x) for x in memdfs.columns]
memdfs.columns = 'share_' + memdfs.columns
memdfs = memdfs.rename(columns = {'share_member_id_':'member_id','share_PostDateMonth_':'PostDateMonth'})

#memdfs.head()

In [ ]:
dfgs1.head()

In [ ]:
memdfs.head()

In [ ]:
#test for dupelicates
memdfs[memdfs.duplicated(subset=['member_id','PostDateMonth'], keep=False)]

In [ ]:
#unstack multi-layer member_id
dfgl1 = dfgl.rename(columns = {'member_id':'ActivityCount'}).reset_index()
dfgl1 = dfgl1.loc[dfgl1['LoanTypeCategory'] != 0]

#create share type catagory-action column
dfgl1['LoanTypeCatAction'] = dfgl1['LoanTypeCategory'] + '-' + dfgl1['TranActionCd']

#credit card flag
dfgl1.loc[dfgl1.LoanTypeCategory.str.contains('Credit Card'),'CC'] = 1

#'top of wallet' for cc per Darren's defintion
dfgl1.loc[((dfgl1.LoanTypeCategory.str.contains('Credit Card')) & (dfgl1.ActivityCount > 9)),'TopCC'] = 1

#create cc history flag
dfgl1['CCLastMo']=dfgl1.groupby(['member_id','LoanTypeCategory','TranActionCd','SourceCodeDescription']).apply(lambda x :x.CC.shift()* (x.PostDateMonth.dt.year*12+x.PostDateMonth.dt.month).diff().eq(1)).replace(0,np.nan).values
dfgl1['LoanBillPayLastMo']=dfgl1.groupby(['member_id','LoanTypeCategory','TranActionCd','SourceCodeDescription']).apply(lambda x :x.BillPay.shift()* (x.PostDateMonth.dt.year*12+x.PostDateMonth.dt.month).diff().eq(1)).replace(0,np.nan).values
dfgl1.loc[((dfgl1.CC==0)&(dfgl1.CCLastMo==1)),'CCStop'] = 1

newcoll = ['member_id', 'PostDateMonth','LoanTypeCatAction', 'BalanceChange', 'PrevAvailBalance',
          'FeeAmount', 'Interest','BillPay','CC','TopCC','CCLastMo','LoanBillPayLastMo','CCStop',
           'Time_diff_Product','Time_diff','Churn']

#unstack and aggregate by member_id, PostDateMonth, ShareTypeCategory, TranActionCd
memdfl = dfgl1[newcoll]
memdfl = memdfl.pivot_table(index=['member_id','PostDateMonth'],
                            columns = 'LoanTypeCatAction',
                            values=['BalanceChange','PrevAvailBalance','FeeAmount','Interest','BillPay',
                                    'Time_diff_Product','Time_diff'],
                            aggfunc=np.sum)

memdfl = memdfl.reset_index()
memdfl.columns = ['_'.join(x) for x in memdfl.columns]
memdfl.columns = 'loan_' + memdfl.columns
memdfl = memdfl.rename(columns = {'loan_member_id_':'member_id','loan_PostDateMonth_':'PostDateMonth'})

#memdfl.head()

In [ ]:
dfgl1.head()

In [ ]:
memdfl.head()

In [ ]:
# merge share and loan data
findf = pd.merge(memdfs,memdfl,how='outer',
                  left_on=['member_id','PostDateMonth'], right_on=['member_id','PostDateMonth'])

findf.head()

In [ ]:
findf.shape

# Portfolio Data

In [ ]:
from pathlib import Path

TRANSACTION_FILE = Path('member_transactions.csv')
if not TRANSACTION_FILE.exists():
    raise FileNotFoundError('Place an anonymized member_transactions.csv file in this directory.')
sq_df = pd.read_csv(TRANSACTION_FILE, parse_dates=['PostDate'])


In [ ]:
from pathlib import Path

MEMBER_FILE = Path('member_status_history.csv')
if not MEMBER_FILE.exists():
    raise FileNotFoundError('Place an anonymized member_status_history.csv file in this directory.')
closed_sq_df = pd.read_csv(MEMBER_FILE, parse_dates=['OpenDate', 'CloseDate'])


In [ ]:
df1 = df_port
df1.columns = map(str.lower, df1.columns)
df1 = df1.fillna(value=np.nan)

In [ ]:
today = pd.to_datetime('today').strftime('%Y-%m-%d')

col_names = df1.columns

for col in col_names:
    if col in df1.filter(regex='count|1m|2m|m3|max|denied|balance|value|amount|deposit', axis=1):
        df1.fillna(0 , inplace = True)
for col in col_names:
    if col in df1.filter(regex='firstope|lastclo', axis=1):
        df1[col].replace('2999-12-31', today)      
    if col in df1.filter(regex='firstope|lastclo', axis=1):
        df1[col] = pd.to_datetime(df1[col], errors = 'coerce').fillna(today)
    if col in df1.filter(regex='flag', axis=1):
        df1[col].astype(bool)

df1.head(100)


In [ ]:
df1['auto_tenure'] = (pd.to_datetime(df1['autolastclosed']) - pd.to_datetime(df1['autofirstopened'])).dt.days.astype(np.int64) // 30.4
df1['personal_tenure'] = (pd.to_datetime(df1['personallastclosed']) - pd.to_datetime(df1['personalfirstopened'])).dt.days.astype(np.int64) // 30.4
df1['other_tenure'] = (pd.to_datetime(df1['otherlastclosed']) - pd.to_datetime(df1['otherfirstopened'])).dt.days.astype(np.int64) // 30.4
df1['cc_tenure'] = (pd.to_datetime(df1['creditcardlastclosed']) - pd.to_datetime(df1['credictcardfirstopened'])).dt.days.astype(np.int64) // 30.4

df1['sav_tenure'] = (pd.to_datetime(df1['savingslastclosed']) - pd.to_datetime(df1['savingsfirstopened'])).dt.days.astype(np.int64) // 30.4
df1['check_tenure'] = (pd.to_datetime(df1['checkinglastclosed']) - pd.to_datetime(df1['checkingfirstopened'])).dt.days.astype(np.int64) // 30.4
df1['cert_tenure'] = (pd.to_datetime(df1['certificatelastclosed']) - pd.to_datetime(df1['certificatefirstopened'])).dt.days.astype(np.int64) // 30.4
df1['ira_tenure'] = (pd.to_datetime(df1['iralastclosed']) - pd.to_datetime(df1['irafirstopened'])).dt.days.astype(np.int64) // 30.4

#df1.head(50)
df1.dtypes

In [ ]:
clean_df = df1[df1.columns.drop(list(df1.filter(regex='firstope|lastclo|date')))]

cols = list(clean_df)
cols.insert(-1, cols.pop(cols.index('churn')))
cols.insert(-2, cols.pop(cols.index('ira_tenure')))
clean_df = clean_df.loc[:, cols]

clean_df.head()

# Merge data sets
## Transactions + Portfolio

In [ ]:
findf['yearmonthsid'] = [''.join(x.split('-')[0:2]) for x in findf.PostDateMonth.astype(str)]
findf['yearmonthsid'] = findf['yearmonthsid'].astype(int)

clean_df['member_id'] = clean_df['member_id']

#merged transaction and portfolio data together
merged = pd.merge(findf,clean_df, left_on=['member_id','yearmonthsid'], 
                     right_on = ['member_id','yearmonthsid'],how='right')
merged = merged.drop(['churn','member_id','identifier'], axis=1)

closed_sq_df['ClosedMonthSid'] = closed_sq_df['ClosedMonthSid'].astype(int)

#merge data with closed data
merged = pd.merge(merged,closed_sq_df[['member_id','ClosedMonth','ClosedMonthSid','Churn']], 
                     left_on=['member_id','yearmonthsid'], 
                     right_on = ['member_id','ClosedMonthSid'],how='left')

merged = pd.merge(merged,closed_sq_df[['member_id','PriorClosedMonth','PriorClosedMonthSid','ChurnIn1Mo']], 
                     left_on=['member_id','yearmonthsid'], 
                     right_on = ['member_id','PriorClosedMonthSid'],how='left')

merged['member_id'] = merged['member_id'].fillna(0)
merged['member_id'] = merged['member_id'].astype(int)
merged['Churn'] = merged['Churn'].fillna(0)
merged['Churn'] = merged['Churn'].astype(int)
merged['directdepositflag'] = merged['directdepositflag'].astype(bool)

#merged_df.directdepositflag = merged_df.loc[merged_df.directdepositflag == 'True'] = 1


merged.head()

In [ ]:
# merged_df.loc[merged_df.Churn == 1]
merged.loc[merged.member_id == 10612]

In [ ]:
dec_df = merged.loc[merged['PostDateMonth'] == '2020-12']

In [ ]:
merged_df = merged.loc[merged['PostDateMonth'] != '2020-12']

# Modeling

In [ ]:
from sklearn.ensemble import RandomForestClassifier #random forrest classifier
from sklearn.model_selection import train_test_split #split data into train and test subsets
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder #code categorical variables

In [ ]:
merged_df.info()

In [ ]:
list(merged_df.columns)

In [ ]:
#set up target variables for modeling
target = merged_df[['Churn']]
target1Mo = merged_df[['ChurnIn1Mo']].fillna(0).astype(int)
merged_df1 = merged_df.drop(['member_id','PostDateMonth','yearmonthsid','Churn','ChurnIn1Mo','ClosedMonth',\
                             'ClosedMonthSid','PriorClosedMonth','PriorClosedMonthSid'], axis=1)

In [ ]:
merged_df1.head()

In [ ]:
merged_df1.dtypes

In [ ]:
#replace all nans with 0
merged_df1 = merged_df1.fillna(0)
merged_df1.head()

In [ ]:
merged_df1.dtypes

### Model Prep

In [ ]:
#importing libraries
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
import statsmodels.api as sm
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.linear_model import RidgeCV, LassoCV, Ridge, Lasso

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report

from sklearn.model_selection import TimeSeriesSplit

# cross validated RF classifier
from sklearn import preprocessing #used for scaling and transforming
from sklearn.pipeline import make_pipeline #used for cross validation
from sklearn.model_selection import GridSearchCV #used for cross validation

In [ ]:
#Using Pearson Correlation
plt.figure(figsize=(12,10))
cor = merged_df.corr()
sns.heatmap(cor, annot=True, cmap=plt.cm.Reds)
plt.show()

In [ ]:
#Correlation with output variable
cor_target = abs(cor["ChurnIn1Mo"])
#Selecting highly correlated features
relevant_features = cor_target[cor_target>0.01]
relevant_features

### Train-Test Split and SMOTE

In [ ]:
# Split data set into Test and Train subsets.  Random State to ensure repeatable outputs.  Stratify to ensure proportional splits
xtrain, xtest, ytrain, ytest = train_test_split(merged_df1, target1Mo, test_size=0.2, random_state = 1, stratify=target1Mo)

print('Training Features Shape:',xtrain.shape)
print('Training Labels Shape:',ytrain.shape)
print('Testing Features Shape:',xtest.shape)
print('Testing Labels Shape:',ytest.shape)

In [ ]:
from imblearn import under_sampling, over_sampling
from imblearn.over_sampling import SMOTE
from imblearn.datasets import make_imbalance

# sm = SMOTE(random_state=42, sampling_strategy=0.6)
smote = SMOTE(sampling_strategy='minority')
xtrain, ytrain = smote.fit_sample(xtrain, ytrain)


In [ ]:
print('Training Features Shape:',xtrain.shape)
print('Training Labels Shape:',ytrain.shape)
print('Testing Features Shape:',xtest.shape)
print('Testing Labels Shape:',ytest.shape)

In [ ]:
print('Training Labels Shape:',ytrain.ChurnIn1Mo.value_counts())

### Recursive Feature Elimination

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import RFECV

In [ ]:
# dtc = DecisionTreeClassifier(random_state=1)
#rfecv = RFECV(estimator=dtc, step=1, cv=StratifiedKFold(10), scoring='accuracy')
#rfecv.fit(xtrain, ytrain)


# rfecv2 = RFECV(estimator=dtc, step=1, cv=StratifiedKFold(10), scoring='accuracy')
# rfecv2.fit(xtrain, ytrain)

In [ ]:
# print('Optimal number of features: {}'.format(rfecv2.n_features_))

In [ ]:
# features = [f for f,s in zip(xtrain.columns, rfecv2.support_) if s]
# print('The selected features are:')
# print ('{}'.format(features))

In [ ]:
# feat_list = rfecv2.get_support() 
# feature_names = list(xtrain.columns.values)
# new_features = xtrain.columns[feat_list]
# xtrain_rfe = pd.DataFrame(xtrain, columns=new_features)
# xtest_rfe = pd.DataFrame(xtest, columns=new_features)

In [ ]:
# xtrain_rfe.head(100)

In [ ]:
# print(xtrain_rfe.shape)

### Logistic Regression Classifier

In [ ]:
LRmodel = LogisticRegression()
LRmodel.fit(xtrain, ytrain)

In [ ]:
# #LR Model with RFE
# lrmodel_rfe = LogisticRegression()
# lrmodel_rfe.fit(xtrain_rfe, ytrain)

In [ ]:
# prediction of LR model on all data
pred_output1 = LRmodel.predict(xtest)

print(confusion_matrix(ytest, pred_output1)) #print confusion matrix
print(classification_report(ytest, LRmodel.predict(xtest))) #print classification report
print(LRmodel.score(xtest, ytest)) #print model accuracy

#### LR Model Confusion Matrix with RFE

In [ ]:
# pred_output1_rfe = lrmodel_rfe.predict(xtest_rfe)
# print(confusion_matrix(ytest, pred_output1_rfe)) #print confusion matrix
# print(classification_report(ytest, lrmodel_rfe.predict(xtest_rfe))) #print classification report
# print(lrmodel_rfe.score(xtest_rfe, ytest)) #print model accuracy

### Random Forest Classifier

In [ ]:
# Instantiate model with 10 decision trees
rf_classifier = RandomForestClassifier(n_estimators = 10, random_state = 1)
# Train the model on training data
rf_classifier.fit(xtrain, ytrain)

In [ ]:
# # Train the model on training data with RFE applied
# rf_classifier_rfe = RandomForestClassifier(n_estimators = 10, random_state = 1)
# rf_classifier_rfe.fit(xtrain_rfe, ytrain)

In [ ]:
pred_output_rf = rf_classifier.predict(xtest)
rfcm = confusion_matrix(ytest, pred_output_rf)
print(rfcm)

In [ ]:
# import shap
# rf_shap_values = shap.KernelExplainer(rf.predict,xtest)

#### RF Confusion Matrix using RFE data

In [ ]:
# pred_output_rf_rfe = rf_classifier_rfe.predict(xtest_rfe)
# rfcm_rfe = confusion_matrix(ytest, pred_output_rf_rfe)
# print(rfcm_rfe)

In [ ]:

#set standardization as a pipeline, more robust
pipeline = make_pipeline(RandomForestClassifier(n_estimators=10, random_state=0))

#define parameters for cross validation testing
hyperparameters = { 'randomforestclassifier__max_features' : ['auto'],
                   'randomforestclassifier__n_estimators' : [10],
                  'randomforestclassifier__max_depth': [None]}

#apply pipeline into cross validation
#set k-fold to 5
rfclf = GridSearchCV(pipeline, hyperparameters, cv=5)
rfclf.fit(xtrain, ytrain)


In [ ]:
# # cross validated RF classifier with RFE

# #define parameters for cross validation testing
# hyperparameters = { 'randomforestclassifier__max_features' : ['auto'],
#                    'randomforestclassifier__n_estimators' : [10],
#                   'randomforestclassifier__max_depth': [None]}

# #apply pipeline into cross validation
# #set k-fold to 5
# rfclf_rfe = GridSearchCV(pipeline, hyperparameters, cv=5)
# rfclf_rfe.fit(xtrain_rfe, ytrain)

In [ ]:
pred_output = rfclf.predict(xtest)
confusion_matrix(ytest, pred_output)

In [ ]:
# pred_output_rf_rfe = rfclf_rfe.predict(xtest_rfe)
# confusion_matrix(ytest, pred_output_rf_rfe)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score #evaluate model performance
print(classification_report(ytest, rfclf.predict(xtest)))

#### RF Model Classification Matrix w/ RFE data

In [ ]:
# from sklearn.metrics import mean_squared_error, r2_score #evaluate model performance
# print(classification_report(ytest, rfclf_rfe.predict(xtest_rfe)))

In [ ]:
score = rfclf.score(xtest, ytest)
print(score)

In [ ]:
results = pd.DataFrame(pred_output)
results.index = xtest.index
results.columns = ["prediction"]

In [ ]:
print("length = ",len(results.loc[results.prediction == 1]))
# results.loc[results.prediction == 1]

In [ ]:
#saves printouts of prediction
output = results.join(merged_df[['member_id']])
output = output[['member_id','prediction']]
output.loc[output.prediction == 1]

In [ ]:
dec_dfmod = dec_df.drop(['member_id','PostDateMonth','yearmonthsid','Churn','ChurnIn1Mo','ClosedMonth',\
                             'ClosedMonthSid','PriorClosedMonth','PriorClosedMonthSid'], axis=1)
dec_dfmod = dec_dfmod.fillna(0)

In [ ]:
dec_dfmod.head()

In [ ]:
import pickle
# save the model to disk
RF_AttritionModel = 'RF_AttritionModel.sav'
pickle.dump(rfclf, open(RF_AttritionModel, 'wb'))

In [ ]:
loaded_model = pickle.load(open(RF_AttritionModel, 'rb'))
dec_result = loaded_model.predict(dec_dfmod)
dec_result = pd.DataFrame(dec_result)
dec_result.index = dec_df.index
dec_result.columns = ["prediction"]

dec_result = dec_result.join(dec_df[['member_id']])
dec_result = dec_result[['member_id','prediction']]
print(len(dec_result.loc[dec_result.prediction == 1]))
dec_result.loc[dec_result.prediction == 1]

In [ ]:
loaded_model.score(xtest, ytest)

In [ ]:
# score = rfclf_rfe.score(xtest_rfe, ytest)
# print(score)

In [ ]:
# from sklearn.model_selection import RandomizedSearchCV
# from sklearn.model_selection import RepeatedStratifiedKFold

# n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)]
# max_features = ['auto']
# max_depth = [int(x) for x in np.linspace(10, 110, num = 11)]
# max_depth.append(None)
# min_samples_split = [2, 5, 10]
# min_samples_leaf = [1, 2, 4]
# bootstrap = [True, False]

# random_grid_rf = {'n_estimators': n_estimators,
# 'max_features': max_features,
# 'max_depth': max_depth,
# 'min_samples_split': min_samples_split,
# 'min_samples_leaf': min_samples_leaf,
# 'bootstrap': bootstrap}

# cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state =1)

# random_rf_mod1 = RandomForestClassifier()
# rf_random = RandomizedSearchCV(estimator = random_rf_mod1, param_distributions = random_grid_rf, n_iter = 20, cv = cv, verbose=2, random_state=1, n_jobs = -1)

# rf_random.fit(xtrain, ytrain)

In [ ]:
# rf_random.best_params_

In [ ]:
# pred_output = rf_random.predict(xtest)
# confusion_matrix(ytest, pred_output)

In [ ]:
# print(classification_report(ytest, rf_random.predict(xtest)))

In [ ]:
# score = rf_random.score(xtest, ytest)
# print(score)

In [ ]:
# feature importance
rffimp = rfclf.best_estimator_._final_estimator.feature_importances_
for name, importance in zip(xtrain, rffimp): 
    print(name, "=", importance)


In [ ]:
# # feature importance w/ RFE data
# rffimp_rfe = rfclf_rfe.best_estimator_._final_estimator.feature_importances_
# for name, importance in zip(xtrain_rfe, rffimp_rfe): 
#     print(name, "=", importance)

In [ ]:
features = xtrain.columns
importances_rf = rfclf.best_estimator_._final_estimator.feature_importances_
indices_rf = np.argsort(importances_rf)

# customisable number 
num_features = 10 

plt.figure(figsize=(24,12))
plt.title('Feature Importances - RF Model')

# only plot the customized number of features
plt.barh(range(num_features), importances_rf[indices_rf[-num_features:]], color='b', align='center')
plt.yticks(range(num_features), [features[i] for i in indices_rf[-num_features:]])
plt.show()

In [ ]:
# features = xtrain_rfe.columns
# importances_rf_rfe = rfclf_rfe.best_estimator_._final_estimator.feature_importances_
# indices_rf_rfe = np.argsort(importances_rf_rfe)

# # customisable number 
# num_features = 10 

# plt.figure(figsize=(24,12))
# plt.title('Feature Importances - RF Model w/ RFE')

# # only plot the customized number of features
# plt.barh(range(num_features), importances_rf_rfe[indices_rf_rfe[-num_features:]], color='b', align='center')
# plt.yticks(range(num_features), [features[i] for i in indices_rf_rfe[-num_features:]])
# plt.show()

In [ ]:
# features = xtrain.columns
# importances_rf = rf_random.best_estimator_._final_estimator.feature_importances_
# indices_rf = np.argsort(importances_rf)

# # customisable number
# num_features = 10

# plt.figure(figsize=(24,12))
# plt.title('Feature Importances - RF Model w randomcv')

# # only plot the customized number of features
# plt.barh(range(num_features), importances_rf[indices_rf[-num_features:]], color='b', align='center')
# plt.yticks(range(num_features), [features[i] for i in indices_rf[-num_features:]])
# plt.show()

### XGBoost Classifier

In [ ]:
# xgboost for feature importance on a classifier problem
from sklearn.datasets import make_classification
from xgboost import XGBClassifier
from matplotlib import pyplot
# define dataset
# xtrain, ytrain = make_classification(n_samples=1000, n_features=10, n_informative=5, n_redundant=5, random_state=1)
# define the model
XGBmodel = XGBClassifier() #verbosity = 0, seed=1 -- can use these parameters if warnings occur
# fit the model
XGBmodel.fit(xtrain, ytrain)


In [ ]:
pred_output = XGBmodel.predict(xtest)
confusion_matrix(ytest, pred_output)

In [ ]:
# xgbmodel_rfe = XGBClassifier()
# xgbmodel_rfe.fit(xtrain_rfe, ytrain)

In [ ]:
print(classification_report(ytest, XGBmodel.predict(xtest)))

In [ ]:
# print(classification_report(ytest, xgbmodel_rfe.predict(xtest_rfe)))

In [ ]:
score = XGBmodel.score(xtest, ytest)
print(score)

In [ ]:
# score = xgbmodel_rfe.score(xtest_rfe, ytest)
# print(score)

In [ ]:
# from sklearn.model_selection import RepeatedStratifiedKFold

# n_estimators = [int(x) for x in np.linspace(start = 100, stop = 500, num = 5)]
# random_grid_xgb = {
# 'n_estimators' :n_estimators,
# 'min_child_weight': [1, 5, 10],
# 'gamma': [0.5, 1, 1.5, 2, 5],
# 'subsample': [0.6, 0.8, 1.0],
# 'colsample_bytree': [0.6, 0.8, 1.0],
# 'max_depth': [3, 4, 5],
# 'max_delta_step':[0,1,2],
# 'learning_rate': [0.001, 0.01, 0.1, 0.2, 0.3]}

# cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state =1)

# random_xgb_mod1 = XGBClassifier()
# xgb_random = RandomizedSearchCV(estimator = random_xgb_mod1, param_distributions = random_grid_xgb, n_iter = 100, cv = cv, verbose=2, random_state=1, n_jobs = -1)

# xgb_random.fit(xtrain, ytrain)

In [ ]:
# xgb_random.best_params_

In [ ]:
# pred_output = xgb_random.predict(xtest)
# confusion_matrix(ytest, pred_output)

In [ ]:
# print(classification_report(ytest, xgb_random.predict(xtest)))

In [ ]:
# score = xgb_random.score(xtest, ytest)
# print(score)

In [ ]:
# feature importance
xgbfimp = XGBmodel.feature_importances_
for name, importance in zip(xtrain, xgbfimp): 
    print(name, "=", importance)

In [ ]:
features = xtrain.columns
importances = XGBmodel.feature_importances_
indices = np.argsort(importances)

# customisable number 
num_features = 10 

plt.figure(figsize=(24,12))
plt.title('Feature Importances - XGB Model')

# only plot the customized number of features
plt.barh(range(num_features), importances[indices[-num_features:]], color='b', align='center')
plt.yticks(range(num_features), [features[i] for i in indices[-num_features:]])
plt.show()

#### XGB feature importances with RFE Data

In [ ]:
# # feature importance
# xgbfimp_rfe = xgbmodel_rfe.feature_importances_
# for name, importance in zip(xtrain_rfe, xgbfimp_rfe): 
#     print(name, "=", importance)

In [ ]:
# features = xtrain_rfe.columns
# importances = xgbmodel_rfe.feature_importances_
# indices = np.argsort(importances)

# # customisable number 
# num_features = 10 

# plt.figure(figsize=(24,12))
# plt.title('Feature Importances - XGB Model w/ RFE Data')

# # only plot the customized number of features
# plt.barh(range(num_features), importances[indices[-num_features:]], color='b', align='center')
# plt.yticks(range(num_features), [features[i] for i in indices[-num_features:]])
# plt.show()

In [ ]:
# features = xtrain.columns
# importances = xgb_random.feature_importances_
# indices = np.argsort(importances)

# # customisable number
# num_features = 10

# plt.figure(figsize=(24,12))
# plt.title('Feature Importances - XGB Model w randomcv')

# # only plot the customized number of features
# plt.barh(range(num_features), importances[indices[-num_features:]], color='b', align='center')
# plt.yticks(range(num_features), [features[i] for i in indices[-num_features:]])
# plt.show()

### SVC Classifier
An SVC model was tested and deemed to be a inefficient application

In [ ]:
# from sklearn.svm import SVC

# # training classifier and evaluating on the whole plane
# SVCclf = SVC(kernel='linear')
# SVCclf.fit(xtrain,ytrain)

In [ ]:
# pred_output = SVCclf.predict(xtest)
# confusion_matrix(ytest, pred_output)


In [ ]:
# print(classification_report(ytest, SVCclf.predict(xtest)))

In [ ]:
# score = SVCclf.score(xtest, ytest)
# print(score)


## Naive Bayes

In [ ]:
#Import Gaussian Naive Bayes model
from sklearn.naive_bayes import GaussianNB

#Create a Gaussian Classifier and train
NBmodel = GaussianNB()
NBmodel.fit(xtrain,ytrain)


In [ ]:
# nbmodel_rfe = GaussianNB()
# nbmodel_rfe.fit(xtrain_rfe,ytrain)

In [ ]:
pred_output = NBmodel.predict(xtest)
confusion_matrix(ytest, pred_output)

In [ ]:
# pred_output_nbrfe = nbmodel_rfe.predict(xtest_rfe)
# confusion_matrix(ytest, pred_output_nbrfe)

In [ ]:
print(classification_report(ytest, NBmodel.predict(xtest)))

#### Naive Bayes Classification Report w/ RFE Data

In [ ]:
# print(classification_report(ytest, nbmodel_rfe.predict(xtest_rfe)))

In [ ]:
score = NBmodel.score(xtest, ytest)
print(score)

In [ ]:
# score = nbmodel_rfe.score(xtest_rfe, ytest)
# print(score)

## Neural Network Classifier

In [ ]:
# Import libraries
import joblib
import sklearn.model_selection
import sklearn.metrics
import sklearn.neural_network

# Create a model
nnmodel = sklearn.neural_network.MLPClassifier(hidden_layer_sizes=(100, ), activation='relu', solver='adam', 
                                             alpha=0.0001, batch_size='auto', learning_rate='constant', learning_rate_init=0.001, power_t=0.5, 
                                             max_iter=1000, shuffle=True, random_state=None, tol=0.0001, verbose=False, warm_start=False, momentum=0.9, 
                                             nesterovs_momentum=True, early_stopping=False, validation_fraction=0.1, beta_1=0.9, beta_2=0.999, epsilon=1e-08, 
                                             n_iter_no_change=10)
nnmodel.fit(xtrain, ytrain)


In [ ]:
# nnmodel_rfe = sklearn.neural_network.MLPClassifier(hidden_layer_sizes=(100, ), activation='relu', solver='adam', 
#                                              alpha=0.0001, batch_size='auto', learning_rate='constant', learning_rate_init=0.001, power_t=0.5, 
#                                              max_iter=1000, shuffle=True, random_state=None, tol=0.0001, verbose=False, warm_start=False, momentum=0.9, 
#                                              nesterovs_momentum=True, early_stopping=False, validation_fraction=0.1, beta_1=0.9, beta_2=0.999, epsilon=1e-08, 
#                                              n_iter_no_change=10)
# nnmodel_rfe.fit(xtrain_rfe, ytrain)

In [ ]:
pred_output = nnmodel.predict(xtest)
confusion_matrix(ytest, pred_output)

In [ ]:
# pred_output_nnrfe = nnmodel_rfe.predict(xtest_rfe)
# confusion_matrix(ytest, pred_output_nnrfe)

In [ ]:
print(classification_report(ytest, nnmodel.predict(xtest)))

#### Neural Net Classification Report w/ RFE Data

In [ ]:
# print(classification_report(ytest, nnmodel_rfe.predict(xtest_rfe)))

In [ ]:
score = nnmodel.score(xtest, ytest)
print(score)

In [ ]:
# score = nnmodel_rfe.score(xtest_rfe, ytest)
# print(score)